In [17]:
# Permite ajustar la anchura de la parte útil de la libreta (reduce los márgenes)
from IPython.core.display import display, HTML
display(HTML("<style>.container{ width:98% }</style>"))

/tmp/ipykernel_823045/1126351303.py:2: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [2]:
## install required packages
!pip install swig
!pip install wrds
!pip install pyportfolioopt
## install finrl library
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

  Cloning https://github.com/AI4Finance-Foundation/FinRL.git to /tmp/pip-req-build-wqmijq_y
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/FinRL.git /tmp/pip-req-build-wqmijq_y
  Resolved https://github.com/AI4Finance-Foundation/FinRL.git to commit 9175621d1235a335a667ee4796d7722e555a9e78
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to /tmp/pip-install-yrcn7_r5/elegantrl_931fedc81ade4468a6a5213c4b2a9846
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git /tmp/pip-install-yrcn7_r5/elegantrl_931fedc81ade4468a6a5213c4b2a9846
  Resolved https://github.com/AI4Finance-Foundation/ElegantRL.git to commit 8ea76afc3e7f1564ae9f0e69e70254116d575fe9
  Preparing metadata (setup.py) ... done


In [3]:
!pip install yfinance

In [18]:
import pandas as pd
import numpy as np
import datetime
import yfinance as yf

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl import config_tickers
from finrl.config import INDICATORS

import itertools

In [19]:
TRAIN_START_DATE = '2009-01-01'
TRAIN_END_DATE = '2020-07-01'
TRADE_START_DATE = '2020-07-01'
TRADE_END_DATE = '2023-05-01'

In [20]:
symbols = [
    'aapl',
    'msft',
    'meta',
    'ibm',
    'hd',
    'cat',
    'amzn',
    'intc',
    't',
    'v'
    'gs'
]

In [21]:
df_raw = YahooDownloader(start_date = TRAIN_START_DATE,
                                end_date = TRADE_END_DATE,
                                ticker_list = symbols).fetch_data()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Shape of DataFrame:  (32958, 8)


In [22]:
df_raw.head()

Price,date,close,high,low,open,volume,tic,day
0,2009-01-02,2.727417,2.736134,2.559415,2.581054,746015200,aapl,4
1,2009-01-02,2.718000,2.726500,2.553500,2.567500,145928000,amzn,4
2,2009-01-02,30.344679,30.389960,28.921564,29.050939,7117200,cat,4
3,2009-01-02,16.148619,16.188774,15.365615,15.439231,14902500,hd,4
4,2009-01-02,47.715294,47.835443,45.814766,45.814766,7905877,ibm,4


In [23]:
fe = FeatureEngineer(use_technical_indicator=True,
                     tech_indicator_list = INDICATORS,
                     use_vix=True,
                     use_turbulence=True,
                     user_defined_feature = False)

processed = fe.preprocess_data(df_raw)

[*********************100%***********************]  1 of 1 completed

Successfully added technical indicators
Shape of DataFrame:  (3604, 8)
Successfully added vix


Successfully added turbulence index


In [10]:
processed

,date,close,high,low,open,volume,tic,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
0,2009-01-02,2.727417,2.736134,2.559415,2.581054,746015200,aapl,4,0.000000,2.947757,2.622185,100.000000,66.666667,100.000000,2.727417,2.727417,39.189999,0.000000
1,2009-01-02,2.718000,2.726500,2.553500,2.567500,145928000,amzn,4,0.000000,2.947757,2.622185,100.000000,66.666667,100.000000,2.718000,2.718000,39.189999,0.000000
2,2009-01-02,30.344679,30.389960,28.921564,29.050939,7117200,cat,4,0.000000,2.947757,2.622185,100.000000,66.666667,100.000000,30.344679,30.344679,39.189999,0.000000
3,2009-01-02,16.148619,16.188774,15.365615,15.439231,14902500,hd,4,0.000000,2.947757,2.622185,100.000000,66.666667,100.000000,16.148619,16.148619,39.189999,0.000000
4,2009-01-02,47.715294,47.835443,45.814766,45.814766,7905877,ibm,4,0.000000,2.947757,2.622185,100.000000,66.666667,100.000000,47.715294,47.715294,39.189999,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28827,2023-04-27,279.383636,280.125034,273.557001,273.747114,3133600,hd,3,0.841059,287.804455,271.096851,48.591925,15.605637,6.854106,276.560941,282.952745,17.030001,5.475535
28828,2023-04-27,116.618004,116.663923,115.231114,116.066924,3204900,ibm,3,-0.754802,122.295066,113.821231,44.911391,-38.238503,1.687847,117.341438,118.950495,17.030001,5.475535
28829,2023-04-27,29.144180,29.232023,27.826543,28.060789,60186200,intc,3,-0.018557,33.393174,28.197083,50.727398,-66.270249,13.386898,30.099059,28.330523,17.030001,5.475535
28830,2023-04-27,299.507843,299.871408,290.095117,290.802548,46462600,msft,3,5.050672,293.828175,271.060958,63.072929,228.848870,42.199524,278.830293,265.935451,17.030001,5.475535


In [24]:
list_ticker = processed["tic"].unique().tolist()
list_date = list(pd.date_range(processed['date'].min(),processed['date'].max()).astype(str))
combination = list(itertools.product(list_date,list_ticker))

processed_full = pd.DataFrame(combination,columns=["date","tic"]).merge(processed,on=["date","tic"],how="left")
processed_full = processed_full[processed_full['date'].isin(processed['date'])]
processed_full = processed_full.sort_values(['date','tic'])

processed_full = processed_full.fillna(0)

In [25]:
# Split the data
train = data_split(processed_full, TRAIN_START_DATE,TRAIN_END_DATE)
trade = data_split(processed_full, TRADE_START_DATE,TRADE_END_DATE)
print(len(train))
print(len(trade))

23144
5688


In [26]:
train

,date,tic,close,high,low,open,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
0,2009-01-02,aapl,2.727417,2.736134,2.559415,2.581054,746015200.0,4.0,0.000000,2.947757,2.622185,100.000000,66.666667,100.000000,2.727417,2.727417,39.189999,0.000000
0,2009-01-02,amzn,2.718000,2.726500,2.553500,2.567500,145928000.0,4.0,0.000000,2.947757,2.622185,100.000000,66.666667,100.000000,2.718000,2.718000,39.189999,0.000000
0,2009-01-02,cat,30.344679,30.389960,28.921564,29.050939,7117200.0,4.0,0.000000,2.947757,2.622185,100.000000,66.666667,100.000000,30.344679,30.344679,39.189999,0.000000
0,2009-01-02,hd,16.148619,16.188774,15.365615,15.439231,14902500.0,4.0,0.000000,2.947757,2.622185,100.000000,66.666667,100.000000,16.148619,16.148619,39.189999,0.000000
0,2009-01-02,ibm,47.715294,47.835443,45.814766,45.814766,7905877.0,4.0,0.000000,2.947757,2.622185,100.000000,66.666667,100.000000,47.715294,47.715294,39.189999,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2892,2020-06-30,hd,223.476715,224.493694,219.533692,220.345495,4524600.0,1.0,2.209611,230.758799,212.732471,56.635875,42.284210,8.960731,220.030361,205.619924,30.430000,4.528901
2892,2020-06-30,ibm,92.458649,92.787844,91.103578,91.249034,4099797.0,1.0,-0.894426,103.374418,86.383610,48.985987,-59.645526,2.646283,94.559642,93.104877,30.430000,4.528901
2892,2020-06-30,intc,53.335049,53.468763,52.185086,52.336630,31542100.0,1.0,-0.500384,57.582326,50.527436,50.231050,-73.270029,8.658016,54.523334,53.486025,30.430000,4.528901
2892,2020-06-30,msft,194.979996,195.832691,189.451852,189.585984,34310300.0,1.0,4.603375,197.144385,173.792845,60.679240,130.633141,29.206539,182.136554,175.005839,30.430000,4.528901


In [27]:
import pickle

with open("train.pkl", "wb") as f:
    pickle.dump(train, f)
    

with open("trade.pkl", "wb") as f:
    pickle.dump(trade, f)

In [13]:
processed_full

,date,tic,close,high,low,open,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
0,2009-01-02,aapl,2.727417,2.736134,2.559415,2.581054,746015200.0,4.0,0.000000,2.947757,2.622185,100.000000,66.666667,100.000000,2.727417,2.727417,39.189999,0.000000
1,2009-01-02,amzn,2.718000,2.726500,2.553500,2.567500,145928000.0,4.0,0.000000,2.947757,2.622185,100.000000,66.666667,100.000000,2.718000,2.718000,39.189999,0.000000
2,2009-01-02,cat,30.344679,30.389960,28.921564,29.050939,7117200.0,4.0,0.000000,2.947757,2.622185,100.000000,66.666667,100.000000,30.344679,30.344679,39.189999,0.000000
3,2009-01-02,hd,16.148619,16.188774,15.365615,15.439231,14902500.0,4.0,0.000000,2.947757,2.622185,100.000000,66.666667,100.000000,16.148619,16.148619,39.189999,0.000000
4,2009-01-02,ibm,47.715294,47.835443,45.814766,45.814766,7905877.0,4.0,0.000000,2.947757,2.622185,100.000000,66.666667,100.000000,47.715294,47.715294,39.189999,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41827,2023-04-27,hd,279.383636,280.125034,273.557001,273.747114,3133600.0,3.0,0.841059,287.804455,271.096851,48.591925,15.605637,6.854106,276.560941,282.952745,17.030001,5.475535
41828,2023-04-27,ibm,116.618004,116.663923,115.231114,116.066924,3204900.0,3.0,-0.754802,122.295066,113.821231,44.911391,-38.238503,1.687847,117.341438,118.950495,17.030001,5.475535
41829,2023-04-27,intc,29.144180,29.232023,27.826543,28.060789,60186200.0,3.0,-0.018557,33.393174,28.197083,50.727398,-66.270249,13.386898,30.099059,28.330523,17.030001,5.475535
41830,2023-04-27,msft,299.507843,299.871408,290.095117,290.802548,46462600.0,3.0,5.050672,293.828175,271.060958,63.072929,228.848870,42.199524,278.830293,265.935451,17.030001,5.475535
